# Beginner Tutorial 2: Understanding the Manifest System

## What You Will Learn

- What declarative programming is and why it matters
- How YAML syntax works (indentation, data types, lists)
- Every section of a `scalable.yaml` manifest
- How to use environment variables for portability
- How overlays customize settings per environment
- How to validate manifests programmatically

## Prerequisites

- Completed notebook 01 (Getting Started)
- `pip install scalable`

In [ ]:
import os
import tempfile

project_dir = tempfile.mkdtemp(prefix="scalable-beginner-02-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## 💡 Key Concept: Declarative vs. Imperative Programming

**Imperative** = step-by-step instructions (HOW to do something):
```python
# Imperative: telling the computer exactly what to do
for i in range(4):
    worker = start_process()
    worker.set_memory('4G')
    worker.connect_to_scheduler(address)
```

**Declarative** = describing desired state (WHAT you want):
```yaml
# Declarative: saying what you need
targets:
  local:
    max_workers: 4
    memory: 4G
```

The manifest is declarative — you describe your desired state and Scalable figures out how.

### Why declarative?
1. **Portable** — same manifest works on laptop, HPC, and cloud
2. **Reproducible** — anyone can recreate your exact setup
3. **Separation of concerns** — scientists declare needs, platform handles infrastructure

## 💡 Key Concept: YAML Syntax

YAML is a human-readable data format. Key rules:

- **Indentation matters** (use spaces, NEVER tabs)
- **Key: value pairs** are the basic unit
- **Nesting** = indentation (2 spaces = child of parent)
- **Lists** use dashes (`- item`)
- **Comments** start with `#`

```yaml
parent:           # This is a map
  child: value    # 2-space indent = nested
  list:           # This will be a list
    - item1
    - item2
```

## Step 1: A Complete Manifest with All Sections

In [ ]:
# Write a full-featured manifest
manifest_content = """\
version: 1
project:
  name: energy-forecast
  default_storage: ./outputs
  local_cache: ./cache

targets:
  local:
    provider: local
    max_workers: 4
    threads_per_worker: 2
    processes: false
    containers: none

  hpc:
    provider: slurm
    queue: batch
    account: GCIMS
    walltime: "04:00:00"

components:
  simulation:
    cpus: 4
    memory: 16G
    tags: [energy, simulation]

  postprocess:
    cpus: 1
    memory: 4G
    tags: [analysis]

tasks:
  run_simulation:
    component: simulation

  aggregate_results:
    component: postprocess

overlays:
  production:
    components:
      simulation:
        memory: 32G
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest_content)

print("Full manifest written. Let's explore each section...")

## Step 2: Understanding Each Section

### `project:` — Metadata
- `name`: Appears in logs and telemetry run IDs
- `default_storage`: Where outputs are saved
- `local_cache`: Where cached results are stored

### `targets:` — WHERE code runs
- Each target has a `provider` key (local, slurm, aws, kubernetes)
- Other keys are provider-specific options
- Switch targets with `--target local` or `--target hpc`

### `components:` — HOW MUCH resources
- `cpus`, `memory`: Resource allocation per worker
- `image`, `mounts`, `env`: Container configuration
- `tags`: Labels for grouping and filtering

### `tasks:` — WHAT work units
- Each task points to a component via `component: name`
- Used when you call `session.submit(func, task="task_name")`

### `overlays:` — Environment-specific customization
- Patches applied on top of base configuration
- Only change what's different (deep merge)

In [ ]:
from scalable.manifest import parse_manifest, validate_manifest

# Parse and validate
# Explanation: parse_manifest reads YAML into Python data structures
# Explanation: validate_manifest checks the structure against schema rules
manifest = parse_manifest("./scalable.yaml")
errors = validate_manifest(manifest)

if errors:
    print("Validation errors:")
    for err in errors:
        print(f"  ✗ {err}")
else:
    print("✓ Manifest is valid!")
    print(f"\nProject name: {manifest.get('project', {}).get('name')}")
    print(f"Targets defined: {list(manifest.get('targets', {}).keys())}")
    print(f"Components defined: {list(manifest.get('components', {}).keys())}")
    print(f"Tasks defined: {list(manifest.get('tasks', {}).keys())}")

## 💡 Key Concept: Overlays

An **overlay** is a set of patches applied on top of base configuration.
Like Photoshop layers — base image + layers that modify specific parts.

**Why overlays?** Instead of maintaining separate manifests for dev/prod
(which drift apart), maintain ONE base + overlays for differences:

- Development: 4 workers, 16G memory
- Production (overlay): same but 32G memory

Apply with: `scalable run ./scalable.yaml --overlay production`

In [ ]:
# Demonstrate overlay concept
# Explanation: Overlays do a "deep merge" — only specified keys change
base_memory = manifest['components']['simulation']['memory']
overlay_memory = manifest.get('overlays', {}).get('production', {}).get('components', {}).get('simulation', {}).get('memory')

print(f"Base manifest: simulation memory = {base_memory}")
print(f"Production overlay: simulation memory = {overlay_memory}")
print(f"\nWith --overlay production, memory becomes {overlay_memory}")
print(f"Everything else (cpus, tags, etc.) stays the same — that's deep merge!")

## Step 3: Validation — Catching Errors Early

Let's see what happens with an invalid manifest:

In [ ]:
# Write a manifest with deliberate errors
bad_manifest = """\
version: 1
project:
  name: test-errors

targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none

components:
  analysis:
    cpus: 1
    memory: 1G

tasks:
  run_analysis:
    component: nonexistent_component
"""

with open("bad_manifest.yaml", "w") as f:
    f.write(bad_manifest)

bad = parse_manifest("./bad_manifest.yaml")
errors = validate_manifest(bad)

print("Validation results for bad manifest:")
if errors:
    for err in errors:
        print(f"  ✗ {err}")
else:
    print("  (No errors detected at parse level)")

print("\n💡 Validation catches errors BEFORE you spend time/money running!")

## 📖 Vocabulary Summary

| Term | Definition |
|------|------------|
| Declarative Programming | Describing WHAT you want, not HOW to achieve it |
| YAML | Human-readable data format using indentation |
| Schema | Rules defining valid structure for data |
| Environment Variables | System-level key-value settings |
| Single Source of Truth | One authoritative location for configuration |
| Overlay | Patches applied on top of base configuration |
| Deep Merge | Recursive merge where only specified keys change |
| Binding | Connecting a task name to a component |
| Parsing | Converting text (YAML) into structured data |
| Validation | Checking that data meets schema rules |

## Next Steps

- **Next notebook:** `03_scaling_strategies.ipynb` — distributed computing fundamentals
- **Try:** Add a third component to the manifest and validate it

In [ ]:
# Cleanup
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print(f"Cleaned up {project_dir}")